# Seed variability in final results

Given the sensitivity of VAE-based architectures to initialization and local minima, the main manuscript reports metrics using the **top-3 seeds** per configuration, following common practice in this setting (see [1]). This is intended to reflect model behavior under proper convergence rather than poor initialization.

To ensure full transparency and to provide a complete view of the variability across runs, this supplementary notebook reports results across **all 10 random seeds** for the final SA/CR experiments in BRCA and LGG. Therefore, this supplementary analysis enables complete inspection of stability and potential upward bias due to seed selection.

[1] Henderson, P., Islam, R., Bachman, P., Pineau, J., Precup, D., & Meger, D. (2018, April). Deep reinforcement learning that matters. In Proceedings of the AAAI conference on artificial intelligence (Vol. 32, No. 1).

In [17]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "results" / "Final_Combinations").exists():
            return candidate
    raise FileNotFoundError("Could not find results/Final_Combinations from current working directory")

PROJECT_ROOT = resolve_project_root()
RESULTS_ROOT = PROJECT_ROOT / "results" / "Final_Combinations"
OUTPUT_DIR = PROJECT_ROOT / "supplementary_tests" / "figs" / "seed_variability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [18]:
def extract_means_from_nested(obj):
    """Extract middle value from tuples like (low, mean, high), recursively."""
    if isinstance(obj, tuple) and len(obj) >= 2:
        try:
            return [float(obj[1])]
        except (TypeError, ValueError):
            return []

    if isinstance(obj, (list, tuple)):
        out = []
        for item in obj:
            out.extend(extract_means_from_nested(item))
        return out

    return []


def index_final_result_files(results_root: Path) -> pd.DataFrame:
    rows = []
    for problem_type, file_name in [("Survival_Analysis", "results.pkl"), ("Competing_Risks", "results_cr.pkl")]:
        base = results_root / problem_type
        if not base.exists():
            continue

        for result_file in base.rglob(file_name):
            rel = result_file.relative_to(base)
            parts = rel.parts
            if len(parts) < 4:
                continue

            dataset = parts[0]
            modality = parts[1]
            run_name = parts[-2]
            experiment = f"{modality} | {run_name}"

            rows.append({
                "problem_type": problem_type,
                "dataset": dataset,
                "modality": modality,
                "run_name": run_name,
                "experiment": experiment,
                "file_path": str(result_file),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["problem_type", "dataset", "modality", "run_name"]).reset_index(drop=True)
    return df


def compute_seed_metrics_from_results(results_dict: dict) -> pd.DataFrame:
    rows = []
    for model_params, seeds_dict in results_dict.items():
        for seed, folds_dict in seeds_dict.items():
            fold_ci = []
            fold_ibs = []

            for fold, fold_data in folds_dict.items():
                if "ci" not in fold_data or "ibs" not in fold_data:
                    continue
                if not fold_data["ci"] or not fold_data["ibs"]:
                    continue

                ci_last = fold_data["ci"][-1]
                ibs_last = fold_data["ibs"][-1]

                ci_vals = extract_means_from_nested(ci_last)
                ibs_vals = extract_means_from_nested(ibs_last)

                if len(ci_vals) == 0 or len(ibs_vals) == 0:
                    continue

                # For CR, ci_vals/ibs_vals contains one value per risk.
                # We aggregate by mean across risks at fold level, then across folds.
                fold_ci.append(float(np.mean(ci_vals)))
                fold_ibs.append(float(np.mean(ibs_vals)))

            if len(fold_ci) == 0 or len(fold_ibs) == 0:
                continue

            rows.append({
                "model_params": str(model_params),
                "seed": int(seed),
                "n_folds_used": int(len(fold_ci)),
                "ci": float(np.mean(fold_ci)),
                "ibs": float(np.mean(fold_ibs)),
            })

    return pd.DataFrame(rows)


def build_seed_dataframe(file_index: pd.DataFrame) -> pd.DataFrame:
    all_rows = []
    failures = []

    for _, row in file_index.iterrows():
        path = Path(row["file_path"])
        try:
            with open(path, "rb") as f:
                results_dict = pickle.load(f)

            seed_df = compute_seed_metrics_from_results(results_dict)
            if seed_df.empty:
                continue

            for col in ["problem_type", "dataset", "modality", "run_name", "experiment", "file_path"]:
                seed_df[col] = row[col]

            all_rows.append(seed_df)

        except Exception as exc:
            failures.append((str(path), str(exc)))

    if failures:
        print("Failed files:")
        for fp, err in failures:
            print(" -", fp, "->", err)

    if len(all_rows) == 0:
        return pd.DataFrame()

    out = pd.concat(all_rows, axis=0, ignore_index=True)
    out = out.sort_values(["problem_type", "dataset", "experiment", "model_params", "seed"]).reset_index(drop=True)
    return out

In [19]:
file_index = index_final_result_files(RESULTS_ROOT)

print("Final result files found:", len(file_index))
seed_df = build_seed_dataframe(file_index)
print("Seed-level rows:", len(seed_df))
print("Unique seeds by problem_type/dataset:")
print(seed_df.groupby(["problem_type", "dataset"])["seed"].nunique().reset_index(name="n_unique_seeds"))

Final result files found: 28
Seed-level rows: 280
Unique seeds by problem_type/dataset:
        problem_type dataset  n_unique_seeds
0    Competing_Risks    brca              10
1    Competing_Risks     lgg              10
2  Survival_Analysis    brca              10
3  Survival_Analysis     lgg              10


In [20]:
from IPython.display import display, Markdown

def topk_mean_local(values, k=3, largest=True):
    vals = np.array(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan
    k = min(k, len(vals))
    idx = np.argsort(vals)
    picked = vals[idx[-k:]] if largest else vals[idx[:k]]
    return float(np.mean(picked))


def build_top3_vs_all10_local(seed_level_df):
    rows = []
    group_cols = ["problem_type", "dataset", "experiment", "model_params"]
    for keys, g in seed_level_df.groupby(group_cols):
        n_seeds = int(g["seed"].nunique())
        if n_seeds != 10:
            continue
        ci_all10 = float(g["ci"].mean())
        ibs_all10 = float(g["ibs"].mean())
        ci_top3 = topk_mean_local(g["ci"].values, k=3, largest=True)
        ibs_top3 = topk_mean_local(g["ibs"].values, k=3, largest=False)
        rows.append({
            "problem_type": keys[0],
            "dataset": keys[1],
            "modality": str(keys[2]).split("|")[0].strip(),
            "experiment": keys[2],
            "model_params": keys[3],
            "n_seeds": n_seeds,
            "ci_top3_reported": ci_top3,
            "ci_all10": ci_all10,
            "ci_gap_top3_minus_all10": ci_top3 - ci_all10,
            "ibs_top3_reported": ibs_top3,
            "ibs_all10": ibs_all10,
            "ibs_gap_all10_minus_top3": ibs_all10 - ibs_top3,
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["problem_type", "dataset", "modality", "experiment", "model_params"]).reset_index(drop=True)
    return out


def build_final_table_both_cancers_sa_cr(top3_vs_all10_df):
    task_map = {"Survival_Analysis": "SA", "Competing_Risks": "CR"}
    df = top3_vs_all10_df.copy()
    df["task"] = df["problem_type"].map(task_map).fillna(df["problem_type"])
    df["dataset"] = df["dataset"].astype(str).str.upper()
    df = df[df["task"].isin(["SA", "CR"]) & df["dataset"].isin(["BRCA", "LGG"])].copy()
    if df.empty:
        return pd.DataFrame()

    final_table = df[[
        "task", "dataset", "modality", "experiment", "model_params", "n_seeds",
        "ci_top3_reported", "ci_all10", "ci_gap_top3_minus_all10",
        "ibs_top3_reported", "ibs_all10", "ibs_gap_all10_minus_top3",
    ]].copy()

    return final_table.sort_values(["dataset", "task", "modality", "experiment", "model_params"]).reset_index(drop=True)


top3_vs_all10_current = build_top3_vs_all10_local(seed_df)
final_table_both_cancers = build_final_table_both_cancers_sa_cr(top3_vs_all10_current)

if final_table_both_cancers.empty:
    print("No rows found for SA/CR and BRCA/LGG.")

final_table_both_cancers = final_table_both_cancers.round(4)

brca_cr_df = final_table_both_cancers[
    (final_table_both_cancers["dataset"] == "BRCA") &
    (final_table_both_cancers["task"] == "CR")
] .reset_index(drop=True)

lgg_cr_df = final_table_both_cancers[
    (final_table_both_cancers["dataset"] == "LGG") &
    (final_table_both_cancers["task"] == "CR")
] .reset_index(drop=True)

brca_sa_df = final_table_both_cancers[
    (final_table_both_cancers["dataset"] == "BRCA") &
    (final_table_both_cancers["task"] == "SA")
] .reset_index(drop=True)

lgg_sa_df = final_table_both_cancers[
    (final_table_both_cancers["dataset"] == "LGG") &
    (final_table_both_cancers["task"] == "SA")
] .reset_index(drop=True)

# Save one CSV per visible table
brca_cr_csv = OUTPUT_DIR / "final_results_BRCA_CR_all_seeds.csv"
lgg_cr_csv = OUTPUT_DIR / "final_results_LGG_CR_all_seeds.csv"
brca_sa_csv = OUTPUT_DIR / "final_results_BRCA_SA_all_seeds.csv"
lgg_sa_csv = OUTPUT_DIR / "final_results_LGG_SA_all_seeds.csv"
 
brca_cr_df.to_csv(brca_cr_csv, index=False)
lgg_cr_df.to_csv(lgg_cr_csv, index=False)
brca_sa_df.to_csv(brca_sa_csv, index=False)
lgg_sa_df.to_csv(lgg_sa_csv, index=False)


print("Saved:", brca_cr_csv)
print("Saved:", lgg_cr_csv)
print("Saved:", brca_sa_csv)
print("Saved:", lgg_sa_csv) 

Saved: /home/alba/snap/snapd-desktop-integration/tfm/samvae-main/supplementary_tests/figs/seed_variability/final_results_BRCA_CR_all_seeds.csv
Saved: /home/alba/snap/snapd-desktop-integration/tfm/samvae-main/supplementary_tests/figs/seed_variability/final_results_LGG_CR_all_seeds.csv
Saved: /home/alba/snap/snapd-desktop-integration/tfm/samvae-main/supplementary_tests/figs/seed_variability/final_results_BRCA_SA_all_seeds.csv
Saved: /home/alba/snap/snapd-desktop-integration/tfm/samvae-main/supplementary_tests/figs/seed_variability/final_results_LGG_SA_all_seeds.csv


# Final Results BRCA-CR

In [21]:
display(brca_cr_df)

,task,dataset,modality,experiment,model_params,n_seeds,ci_top3_reported,ci_all10,ci_gap_top3_minus_all10,ibs_top3_reported,ibs_all10,ibs_gap_all10_minus_top3
0,CR,BRCA,clinical,clinical | 5_folds_512_batch_size,[10]_[500],10,0.6249,0.6023,0.0226,0.2434,0.2535,0.0101
1,CR,BRCA,clinical_omic_cnv_omic_RNAseq,clinical_omic_cnv_omic_RNAseq | 5_folds_512_ba...,"[10, 5, 50]_[500, 50, 50]",10,0.5943,0.5756,0.0187,0.2504,0.2619,0.0116
2,CR,BRCA,clinical_omic_cnv_omic_RNAseq_wsi_patches_10_p...,clinical_omic_cnv_omic_RNAseq_wsi_patches_10_p...,"[10, 5, 50, 50]_[500, 50, 50, [16, 32, 64]]",10,0.6067,0.5815,0.0252,0.2432,0.2537,0.0105
3,CR,BRCA,clinical_wsi_patches_10_patch,clinical_wsi_patches_10_patch | 5_folds_50_bat...,"[10, 50]_[500, [16, 32, 64]]",10,0.6102,0.5854,0.0248,0.2380,0.2474,0.0094
4,CR,BRCA,omic_cnv_omic_RNAseq,omic_cnv_omic_RNAseq | 5_folds_512_batch_size,"[5, 50]_[50, 50]",10,0.5920,0.5626,0.0294,0.2452,0.2555,0.0103
5,CR,BRCA,omic_cnv_omic_RNAseq_wsi_patches_10_patch,omic_cnv_omic_RNAseq_wsi_patches_10_patch | 5_...,"[5, 50, 50]_[50, 50, [16, 32, 64]]",10,0.5956,0.5749,0.0206,0.2457,0.2579,0.0122
6,CR,BRCA,wsi_patches_10_patch,wsi_patches_10_patch | 5_folds_50_batch_size,"[50]_[[16, 32, 64]]",10,0.5984,0.5794,0.0190,0.2388,0.2469,0.0081


# Final Results LGG-CR

In [22]:
display(lgg_cr_df)

,task,dataset,modality,experiment,model_params,n_seeds,ci_top3_reported,ci_all10,ci_gap_top3_minus_all10,ibs_top3_reported,ibs_all10,ibs_gap_all10_minus_top3
0,CR,LGG,clinical,clinical | 5_folds_512_batch_size,[5]_[75],10,0.5716,0.5608,0.0108,0.3439,0.3512,0.0073
1,CR,LGG,clinical_omic_adn_omic_cnv_omic_miRNA,clinical_omic_adn_omic_cnv_omic_miRNA | 5_fold...,"[5, 5, 5, 5]_[75, 50, 500, 50]",10,0.5586,0.5472,0.0114,0.3443,0.3551,0.0108
2,CR,LGG,clinical_omic_adn_omic_cnv_omic_miRNA_wsi_patc...,clinical_omic_adn_omic_cnv_omic_miRNA_wsi_patc...,"[5, 5, 5, 5, 5]_[75, 50, 500, 50, [16, 32, 64]]",10,0.5539,0.5398,0.0142,0.3560,0.3658,0.0099
3,CR,LGG,clinical_wsi_patches_5_patch,clinical_wsi_patches_5_patch | 5_folds_105_bat...,"[5, 5]_[75, [16, 32, 64]]",10,0.5684,0.5539,0.0145,0.3618,0.3708,0.0090
4,CR,LGG,omic_adn_omic_cnv_omic_miRNA,omic_adn_omic_cnv_omic_miRNA | 5_folds_512_bat...,"[5, 5, 5]_[50, 500, 50]",10,0.5585,0.5455,0.0131,0.3403,0.3550,0.0146
5,CR,LGG,omic_adn_omic_cnv_omic_miRNA_wsi_patches_5_patch,omic_adn_omic_cnv_omic_miRNA_wsi_patches_5_pat...,"[5, 5, 5, 5]_[50, 500, 50, [16, 32, 64]]",10,0.5664,0.5434,0.0231,0.3646,0.3683,0.0036
6,CR,LGG,wsi_patches_5_patch,wsi_patches_5_patch | 5_folds_105_batch_size,"[5]_[[16, 32, 64]]",10,0.5715,0.5526,0.0189,0.3566,0.3629,0.0063


# Final Results BRCA-SA

In [23]:
display(brca_sa_df)

,task,dataset,modality,experiment,model_params,n_seeds,ci_top3_reported,ci_all10,ci_gap_top3_minus_all10,ibs_top3_reported,ibs_all10,ibs_gap_all10_minus_top3
0,SA,BRCA,clinical,clinical | 5_folds_1_batch_size,[10]_[100],10,0.7106,0.6753,0.0353,0.1732,0.1921,0.0189
1,SA,BRCA,clinical_omic_cnv,clinical_omic_cnv | 5_folds_1_batch_size,"[10, 5]_[100, 5]",10,0.6601,0.6327,0.0274,0.2011,0.2159,0.0148
2,SA,BRCA,clinical_omic_cnv_wsi_patches_15_patch,clinical_omic_cnv_wsi_patches_15_patch | 5_fol...,"[10, 5, 5]_[100, 5, [32, 64, 128]]",10,0.6290,0.6085,0.0205,0.1898,0.1964,0.0065
3,SA,BRCA,clinical_wsi_patches_15_patch,clinical_wsi_patches_15_patch | 5_folds_1_batc...,"[10, 5]_[100, [32, 64, 128]]",10,0.6515,0.6220,0.0295,0.1920,0.2022,0.0102
4,SA,BRCA,omic_cnv,omic_cnv | 5_folds_512_batch_size,[5]_[5],10,0.6014,0.5815,0.0199,0.1907,0.2031,0.0124
5,SA,BRCA,omic_cnv_wsi_patches_15_patch,omic_cnv_wsi_patches_15_patch | 5_folds_1_batc...,"[5, 5]_[5, [32, 64, 128]]",10,0.6117,0.5770,0.0347,0.1923,0.2029,0.0106
6,SA,BRCA,wsi_patches_15_patch,wsi_patches_15_patch | 5_folds_1_batch_size,"[5]_[[32, 64, 128]]",10,0.5928,0.5767,0.0161,0.2035,0.2079,0.0044


# Final Results LGG-SA

In [24]:
display(lgg_sa_df)

,task,dataset,modality,experiment,model_params,n_seeds,ci_top3_reported,ci_all10,ci_gap_top3_minus_all10,ibs_top3_reported,ibs_all10,ibs_gap_all10_minus_top3
0,SA,LGG,clinical,clinical | 5_folds_1_batch_size,[10]_[10],10,0.7253,0.6968,0.0285,0.1876,0.2009,0.0133
1,SA,LGG,clinical_omic_adn,clinical_omic_adn | 5_folds_1_batch_size,"[10, 5]_[10, 50]",10,0.7032,0.6729,0.0303,0.1888,0.1970,0.0082
2,SA,LGG,clinical_omic_adn_wsi_patches_10_patch,clinical_omic_adn_wsi_patches_10_patch | 5_fol...,"[10, 5, 5]_[10, 50, [8, 16, 32]]",10,0.7293,0.6925,0.0368,0.1920,0.2001,0.0081
3,SA,LGG,clinical_wsi_patches_10_patch,clinical_wsi_patches_10_patch | 5_folds_1_batc...,"[10, 5]_[10, [8, 16, 32]]",10,0.7398,0.7145,0.0253,0.1903,0.1962,0.0059
4,SA,LGG,omic_adn,omic_adn | 5_folds_1_batch_size,[5]_[50],10,0.5913,0.5628,0.0284,0.1968,0.2010,0.0042
5,SA,LGG,omic_adn_wsi_patches_10_patch,omic_adn_wsi_patches_10_patch | 5_folds_1_batc...,"[5, 5]_[50, [8, 16, 32]]",10,0.5850,0.5701,0.0149,0.1986,0.2039,0.0053
6,SA,LGG,wsi_patches_10_patch,wsi_patches_10_patch | 5_folds_1_batch_size,"[5]_[[8, 16, 32]]",10,0.5672,0.5501,0.0171,0.1952,0.1980,0.0028
